# 04. Dataset Splitting

This notebook covers the fourth step in a typical QSAR workflow:
- Splitting the dataset into training and test sets
- Using appropriate splitting strategies to ensure proper validation
- Visualizing the distribution of compounds and activities across splits

ProQSAR supports various splitting methods:
- Random splitting
- Scaffold-based splitting (preserves chemical diversity)
- Stratified splitting (preserves activity distribution)
- Kennard-Stone algorithm (maximizes coverage of feature space)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from proqsar.Data.Splitter.random_splitter import RandomSplitter
from proqsar.Data.Splitter.scaffold_splitter import ScaffoldSplitter
from proqsar.Data.Splitter.stratified_random_splitter import StratifiedRandomSplitter
from proqsar.Config.config import Config
from sklearn.decomposition import PCA

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 4.1 Load Dataset with Selected Features

Load the dataset with selected features from the previous step.

In [ ]:
# Load the dataset with selected features
data_path = '../Project/selected_features_data.csv'
df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape}")
print(f"Number of compounds: {len(df)}")
print(f"Number of features: {df.shape[1] - 2}")
df.head()

## 4.2 Configure Data Splitter

Choose a splitting strategy based on your modeling needs.

In [ ]:
# Configure splitter
# Options: RandomSplitter, ScaffoldSplitter, StratifiedRandomSplitter

config = Config(
    splitter={
        "split_type": "random",  # Options: "random", "scaffold", "stratified"
        "test_size": 0.2,  # 20% for test set
        "random_state": 42,
    }
)

print("Splitter configured!")
print(f"Split type: {config.splitter_config['split_type']}")
print(f"Test size: {config.splitter_config['test_size']}")

## 4.3 Split the Dataset

Split the dataset into training and test sets.

In [ ]:
# Initialize splitter based on configuration
split_type = config.splitter_config['split_type']

if split_type == "random":
    splitter = RandomSplitter(
        test_size=config.splitter_config['test_size'],
        random_state=config.splitter_config['random_state']
    )
elif split_type == "scaffold":
    splitter = ScaffoldSplitter(
        smiles_col="Smiles",
        test_size=config.splitter_config['test_size'],
        random_state=config.splitter_config['random_state']
    )
elif split_type == "stratified":
    splitter = StratifiedRandomSplitter(
        activity_col="pChEMBL",
        test_size=config.splitter_config['test_size'],
        random_state=config.splitter_config['random_state'],
        n_bins=5
    )
else:
    raise ValueError(f"Unknown split type: {split_type}")

print(f"Splitter initialized: {splitter.__class__.__name__}")

In [ ]:
# Perform the split
train_idx, test_idx = splitter.split(df)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"\nDataset split complete!")
print(f"Training set size: {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test set size: {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")

## 4.4 Analyze Split Quality

Visualize and analyze the distribution of compounds and activities across splits.

In [ ]:
# Compare activity distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograms
axes[0].hist(df_train['pChEMBL'], bins=20, alpha=0.6, label='Train', edgecolor='black')
axes[0].hist(df_test['pChEMBL'], bins=20, alpha=0.6, label='Test', edgecolor='black')
axes[0].set_xlabel('pChEMBL Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Activity Distribution')
axes[0].legend()

# Box plots
data_to_plot = [df_train['pChEMBL'], df_test['pChEMBL']]
axes[1].boxplot(data_to_plot, labels=['Train', 'Test'])
axes[1].set_ylabel('pChEMBL Value')
axes[1].set_title('Activity Distribution Comparison')

# Kernel density estimation
df_train['pChEMBL'].plot(kind='density', ax=axes[2], label='Train', alpha=0.7)
df_test['pChEMBL'].plot(kind='density', ax=axes[2], label='Test', alpha=0.7)
axes[2].set_xlabel('pChEMBL Value')
axes[2].set_ylabel('Density')
axes[2].set_title('Activity Density Plot')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Statistical comparison
print("Activity Statistics Comparison:")
print("="*60)
print("\nTraining Set:")
print(df_train['pChEMBL'].describe())
print("\nTest Set:")
print(df_test['pChEMBL'].describe())

# Statistical test (Kolmogorov-Smirnov test)
from scipy.stats import ks_2samp
statistic, pvalue = ks_2samp(df_train['pChEMBL'], df_test['pChEMBL'])
print("\nKolmogorov-Smirnov Test:")
print(f"Statistic: {statistic:.4f}")
print(f"P-value: {pvalue:.4f}")
if pvalue > 0.05:
    print("The distributions are similar (p > 0.05)")
else:
    print("The distributions are significantly different (p < 0.05)")

In [ ]:
# Visualize feature space coverage using PCA
feature_cols = [col for col in df.columns if col not in ['Smiles', 'pChEMBL']]
X = df[feature_cols].values
X_train = df_train[feature_cols].values
X_test = df_test[feature_cols].values

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

# Plot PCA
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], alpha=0.6, label='Train', s=50)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], alpha=0.6, label='Test', s=50)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Feature Space Coverage (PCA)')
plt.legend()
plt.grid(True, alpha=0.3)

# Color by activity
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                     c=df['pChEMBL'], cmap='viridis', alpha=0.6, s=50)
plt.colorbar(scatter, label='pChEMBL')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Feature Space Colored by Activity')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

## 4.5 Save Split Datasets

Save the training and test sets for use in model building.

In [ ]:
# Save training and test sets
train_path = '../Project/train_data.csv'
test_path = '../Project/test_data.csv'

df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)

print(f"Training set saved to: {train_path}")
print(f"Training set shape: {df_train.shape}")
print(f"\nTest set saved to: {test_path}")
print(f"Test set shape: {df_test.shape}")

## 4.6 Summary

Summarize the dataset splitting process.

In [ ]:
print("="*60)
print("DATASET SPLITTING SUMMARY")
print("="*60)
print(f"Total compounds: {len(df)}")
print(f"Training set: {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test set: {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")
print(f"\nSplitting method: {split_type}")
print(f"\nTraining set activity: {df_train['pChEMBL'].mean():.3f} ± {df_train['pChEMBL'].std():.3f}")
print(f"Test set activity: {df_test['pChEMBL'].mean():.3f} ± {df_test['pChEMBL'].std():.3f}")
print(f"\nKS test p-value: {pvalue:.4f}")
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"\nData saved to:")
print(f"  - {train_path}")
print(f"  - {test_path}")
print("\nDatasets are ready for model building!")
print("="*60)

## Next Steps

The dataset has been split into training and test sets. The next notebook (05_model_building.ipynb) will:
- Build QSAR models using various algorithms (MLR, PLS, Random Forest, etc.)
- Train models on the training set
- Save trained models for validation